---
title: "Análise Performance de Campanhas"
format:
  html:
    html-math-method: katex
    embed-resources: true
    code-fold: true
---

# **Objetivos da Análise**

* Os dados utilizados foram retirados do Kaggle no seguinte link: https://www.kaggle.com/datasets/amirmotefaker/ab-testing-dataset
* A ideia principal do estudo é simular o trabalho de um Analista/Cientista de dados de uma empresa, para aferir se a mudança entre a campanha surtiu efeito ou não, e se sim, em qual escopo. Bem como recomendar ao setor de marketing qual campanha foi mais efetiva para o quê, bem como outras recomendações, e análise de indicadores relevantes para campanhas online.
* Desse modo, o estudo consiste em:
    1. Análise exploratória e descritiva
    2. Análise de efetividade por meio dos testes de hipóteses e indicadores

* Além disso, como não há a disponibilidade dos dados ao nível de cliente, os indicadores serão calculados por meio dos agregados, o que faz com que determinadas métricas sofram algumas alterações em suas fórmulas.
Devido à natureza agregada dos dados fornecidos, a inferência estatística focar-se-á em testes de proporção (Teste Z), amparados pelo Teorema do Limite Central, dado o grande volume amostral (N). Testes sobre variáveis contínuas (como Ticket Médio) foram descartados, pois a agregação prévia impossibilita o cálculo da variância individual e a verificação de premissas paramétricas como normalidade e homocedasticidade.



## 1 - Tratamento dos dados

In [1]:
# Importando as bibliotecas
import pandas as pd
import numpy as np
import plotly.express as px
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# Carregando as amostras de controle e de teste para campanhas
df_controle = pd.read_excel('control_group.xlsm')
df_teste = pd.read_excel("test_group.xlsm")

In [3]:
# Analisando o nome, quantidade de colunas e os tipos de variaveis.
df_controle.info()
df_teste.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Campaign Name        30 non-null     object 
 1   Date                 30 non-null     object 
 2   Spend [USD]          30 non-null     int64  
 3   # of Impressions     29 non-null     float64
 4   Reach                29 non-null     float64
 5   # of Website Clicks  29 non-null     float64
 6   # of Searches        29 non-null     float64
 7   # of View Content    29 non-null     float64
 8   # of Add to Cart     29 non-null     float64
 9   # of Purchase        29 non-null     float64
dtypes: float64(7), int64(1), object(2)
memory usage: 2.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Campaign Name        30 non-n


* Nota-se que em um dos controles obtivemos valores nulos, dessa forma substituiremos esses valores nulos por zero
* Também destaca-se que devemos converter a coluna data de cada amostra para o tipo de dado para data
* Além disso, serão traduzidas e padronizadas as colunas da amostra

In [4]:
df_controle.head(10)

,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase
0,Control Campaign,1.08.2019,2280,82702.0,56930.0,7016.0,2290.0,2159.0,1819.0,618.0
1,Control Campaign,2.08.2019,1757,121040.0,102513.0,8110.0,2033.0,1841.0,1219.0,511.0
2,Control Campaign,3.08.2019,2343,131711.0,110862.0,6508.0,1737.0,1549.0,1134.0,372.0
3,Control Campaign,4.08.2019,1940,72878.0,61235.0,3065.0,1042.0,982.0,1183.0,340.0
4,Control Campaign,5.08.2019,1835,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Control Campaign,6.08.2019,3083,109076.0,87998.0,4028.0,1709.0,1249.0,784.0,764.0
6,Control Campaign,7.08.2019,2544,142123.0,127852.0,2640.0,1388.0,1106.0,1166.0,499.0
7,Control Campaign,8.08.2019,1900,90939.0,65217.0,7260.0,3047.0,2746.0,930.0,462.0
8,Control Campaign,9.08.2019,2813,121332.0,94896.0,6198.0,2487.0,2179.0,645.0,501.0
9,Control Campaign,10.08.2019,2149,117624.0,91257.0,2277.0,2475.0,1984.0,1629.0,734.0


In [5]:
df_teste.head(10)

,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase
0,Test Campaign,1.08.2019,3008,39550,35820,3038,1946,1069,894,255
1,Test Campaign,2.08.2019,2542,100719,91236,4657,2359,1548,879,677
2,Test Campaign,3.08.2019,2365,70263,45198,7885,2572,2367,1268,578
3,Test Campaign,4.08.2019,2710,78451,25937,4216,2216,1437,566,340
4,Test Campaign,5.08.2019,2297,114295,95138,5863,2106,858,956,768
5,Test Campaign,6.08.2019,2458,42684,31489,7488,1854,1073,882,488
6,Test Campaign,7.08.2019,2838,53986,42148,4221,2733,2182,1301,890
7,Test Campaign,8.08.2019,2916,33669,20149,7184,2867,2194,1240,431
8,Test Campaign,9.08.2019,2652,45511,31598,8259,2899,2761,1200,845
9,Test Campaign,10.08.2019,2790,95054,79632,8125,2312,1804,424,275


In [6]:
# Tratamento dos dados
# Criação de dicionario para substição dos nomes
Padronizacao_colunas = {'Campaign Name': 'Nome Campanha', 'Date': 'Data', 'Spend [USD]': 'Gasto em Dólar', '# of Impressions': 'Qtd Impressões', 'Reach': 'Alcance',
       '# of Website Clicks': 'Qtd de clicks no Website', '# of Searches': 'Qtd de procura', '# of View Content':'Qtd de visualização',
       '# of Add to Cart': 'Qtd adicionado ao Carrinho', '# of Purchase': 'Qtd de Compras'    
}
# Alterar o nome das colunas permanentemente a partir deste ponto
df_controle.rename(columns=Padronizacao_colunas, inplace=True)
df_teste.rename(columns=Padronizacao_colunas, inplace=True)
# Substitui os valores NULL por 0
df_controle.fillna(0,inplace=True)
# Formatando a data
df_controle['Data'] = pd.to_datetime(df_controle['Data'],format='mixed',dayfirst=True)
df_teste['Data'] = pd.to_datetime(df_teste['Data'],format='mixed',dayfirst=True)

In [7]:
# Consolidando ambas as amostras em um mesmo DataFrame
df_consolidado = pd.concat([df_controle,df_teste],ignore_index=True)
# Agregando os resultados em um DataFrame
tabela_agregado = df_consolidado.groupby('Nome Campanha').sum(numeric_only=True).reset_index()

## 2 - Teste de Sanidade - Sample Ratio Mismatch
Antes de realizar as análises exploratórias e os testes de hipótese, será realizado o teste de sanidade dos dados para verificação se o tráfego foi dividido em 50/50, ou seja 50% para controle e 50% para teste afim de descobrir se há SRM (*Sample Ratio Mismatch*).
Para o teste em questão será utilizado o teste Qui-Quadrado de Aderência

In [8]:

from scipy.stats import chisquare
# Cálculo do Teste qui-quadrado
alcance_controle = tabela_agregado.loc[tabela_agregado['Nome Campanha'] == 'Control Campaign', 'Alcance'].values[0]
alcance_teste = tabela_agregado.loc[tabela_agregado['Nome Campanha'] == 'Test Campaign', 'Alcance'].values[0]

observados = [alcance_controle, alcance_teste]
total_usuarios = sum(observados)

# Assumimos que o planejamento original do A/B test era dividir o tráfego ao meio.
esperados = [total_usuarios / 2, total_usuarios / 2]

chi2_stat, p_valor_srm = chisquare(f_obs=observados, f_exp=esperados)

print(f"=== TESTE DE SANIDADE: SRM (Sample Ratio Mismatch) ===")
print(f"Tráfego Controle: {observados[0]:,.0f} ({observados[0]/total_usuarios:.2%})")
print(f"Tráfego Teste:    {observados[1]:,.0f} ({observados[1]/total_usuarios:.2%})")
print(f"Estatística Chi2: {chi2_stat:.4f} | P-valor: {p_valor_srm:.4e}\n")

alpha_srm = 0.05
if p_valor_srm < alpha_srm:
    print("REJEITAMOS H0: SRM Detectado")
    print("A distribuição de tráfego é estatisticamente diferente do esperado (50/50).")
    print("O teste possui um viés de amostragem que deve ser considerado na análise das taxas.")
else:
    print("NÃO REJEITAMOS HO: Sem indícios de SRM.")
    print("O tráfego está estatisticamente balanceado.")

=== TESTE DE SANIDADE: SRM (Sample Ratio Mismatch) ===
Tráfego Controle: 2,576,503 (61.62%)
Tráfego Teste:    1,604,747 (38.38%)
Estatística Chi2: 225843.8801 | P-valor: 0.0000e+00

REJEITAMOS H0: SRM Detectado
A distribuição de tráfego é estatisticamente diferente do esperado (50/50).
O teste possui um viés de amostragem que deve ser considerado na análise das taxas.


Com base no teste, **há presença de SRM**, desse modo a plataforma utilizada para disparar os anúncios privilegiou mais o tráfego da campanha controle de forma mais precoce.Logo, inflando artificialmente a vantagem para a campanha Controle.<br>
Com isso, espera-se que os testes de hipótese possam ser impactados de alguma forma, principalmente na métrica que o controle melhor desempenhou. **Desse modo, as inferências causais sobre cada campanha possa ser impactada de forma significativa**<br>
Ademais, segue-se para a seção 3, aonde será realizada a análise exploratória entre as campanhas. Bem como, analisar possíveis motivos pelo qual a plataforma de anúncio em questão privilegiou a campanha controle.

## 3 - Análise exploratoria e descritiva

In [9]:
# Análise do somatório das campanhas
tabela_agregado

,Nome Campanha,Gasto em Dólar,Qtd Impressões,Alcance,Qtd de clicks no Website,Qtd de procura,Qtd de visualização,Qtd adicionado ao Carrinho,Qtd de Compras
0,Control Campaign,68653,3177233.0,2576503.0,154303.0,64418.0,56370.0,37700.0,15161.0
1,Test Campaign,76892,2237544.0,1604747.0,180970.0,72569.0,55740.0,26446.0,15637.0



* Nota-se que foi gasto mais no total com a campanha de Teste do que com a Controle. Evidencia-se também que a campanha controle obteve um maior número de variáveis cujo valor agregado é ligeiramente maior se comparado à campanha de teste. **Porém, na seção 4 será testado se isso é estatisticamente significativo ou não e para quais variáveis.**
* Com base na tabela acima, temos a impressão de que a campanha controle tem um desempenho ligeiramente melhor se comparada à campanha de teste.
* Ademais, conseguimos criar outras variáveis com base nas já disponibilizadas, tais como:

    * CPC (Custo por Clique): Gasto em Dólar da campanha / Qtd de Cliques no Website
    * CPA (Custo por Aquisição): Gasto em Dólar / Qtd de Compras
    * CTR (Taxa de Clique): Qtd de clicks no Website / Qtd Impressões
    * Drop-off Rate: Qtd de visualização / Qtd de clicks no Website
    * CR ou Taxa de Conversão: Qtd de Compras / Qtd de clicks no Website
    * Taxa de Adição ao Carrinho: Qtd adicionado ao Carrinho / Qtd de visualização
    * Taxa de Abandono de Carrinho: 1 - (Qtd de Compras / Qtd adicionado ao Carrinho)

Entretanto, ressalta-se que para validar que houve uma diferença estatística significativa entre cada campanha serão realizados testes Z para duas proporções amparados pelo Teorema do Limite Central devido à grande amostra (quando se olha os dados de maneira agregada).

In [10]:
# Gráfico com a quantidade de impressões geradas pelos anúncios de cada campanha
fig = px.line(
    df_consolidado,x='Data',y='Qtd Impressões', color='Nome Campanha',
    title = 'Quantidade de impressões geradas por cada campanha ao longo do mês de Agosto',
    labels= {'Data':'Dia','Qtd Impressões':'Quantidade de Impressões','Nome Campanha': 'Tipo da Campanha'}
)
fig.show(renderer='notebook_connected')

O gráfico acima representa a quantidade de impressões que ambas as campanhas obtiveram durante o mês de agosto. Podem-se notar algumas informações relevantes, tais como:

* A campanha de controle, ao que parece, obteve mais impressões na maioria dos dias
* A campanha de controle teve um dia sem nenhum tipo de impressão, isso pode impactar os testes estatísticos


In [11]:
# Cálculo dos KPI's para análise gráfica mensal
df_consolidado['CR'] = df_consolidado['Qtd de Compras'] / df_consolidado['Qtd de clicks no Website']
df_consolidado['CTR'] = df_consolidado['Qtd de clicks no Website'] / df_consolidado['Qtd Impressões']

#Criando um dicionário
config_graficos = {
    'CR': {
        'titulo': 'Taxa de Conversão (CR) de cada campanha ao longo de Agosto',
        'label_y': 'Taxa de Conversão'
    },
    'CTR': {
        'titulo': 'Taxa de Clique (CTR) de cada campanha ao longo de Agosto',
        'label_y': 'Taxa de Clique'
    }
}

# Loop para os gráficos
for coluna, textos in config_graficos.items():
    fig = px.line(
        df_consolidado, x='Data', y=coluna, color= 'Nome Campanha',
        title = textos['titulo'],
        labels = {
            'Data':'Dia',
            coluna: textos['label_y'],
            'Nome Campanha': 'Tipo Campanha'
        }
    )
    fig.update_layout(yaxis_tickformat='.2%')

    fig.show(renderer='notebook_connected')


In [12]:
# Cálculo dos principais KPI's
## Métricas de custo
# Custo por Clique
tabela_agregado['CPC'] = tabela_agregado['Gasto em Dólar'] / tabela_agregado['Qtd de clicks no Website']
# Custo por Aquisição
tabela_agregado['CPA'] = tabela_agregado['Gasto em Dólar'] / tabela_agregado['Qtd de Compras']
## Métricas de Engajamento
# Taxa de Clique
tabela_agregado['CTR'] = tabela_agregado['Qtd de clicks no Website'] / tabela_agregado['Qtd Impressões']
# Frequência
tabela_agregado['Frequência'] = tabela_agregado['Qtd Impressões'] / tabela_agregado['Alcance']
# Drop-off
tabela_agregado['Drop-off'] = tabela_agregado['Qtd de visualização'] / tabela_agregado['Qtd de clicks no Website']
## Métricas de Conversão
# Taxa de adição no carrinho
tabela_agregado['Taxa carrinho'] = tabela_agregado['Qtd adicionado ao Carrinho'] / tabela_agregado['Qtd de visualização']
# Taxa de Abandono de Carrinho
tabela_agregado['Taxa abandono'] = 1 - (tabela_agregado['Qtd de Compras'] / tabela_agregado['Qtd adicionado ao Carrinho'])
# Taxa de Conversão ou CR
tabela_agregado['CR'] = tabela_agregado['Qtd de Compras'] / tabela_agregado['Qtd de clicks no Website']


In [13]:
# Alterando a visualização do dataframe para exibição dos resultados.
regras_formatacao = {
    # Grupo 1: Moeda (Adiciona o símbolo $, separador de milhar com vírgula e 2 casas decimais)
    'Gasto em Dólar': '$ {:,.2f}',
    'CPC': '$ {:,.2f}',
    'CPA': '$ {:,.2f}',
    
    # Grupo 2: Quantidades (Adiciona separador de milhar com vírgula e remove casas decimais)
    'Qtd Impressões': '{:,.0f}',
    'Alcance': '{:,.0f}',
    'Qtd de clicks no Website': '{:,.0f}',
    'Qtd de procura': '{:,.0f}',
    'Frequência': '{:,.2f}',
    'Qtd de visualização': '{:,.0f}',
    'Qtd adicionado ao Carrinho': '{:,.0f}',
    'Qtd de Compras': '{:,.0f}',
    
    # Grupo 3: Percentuais (Multiplica por 100, adiciona o % e deixa 2 casas decimais)
    'CTR': '{:.2%}',
    'Drop-off': '{:.2%}',
    'CR': '{:.2%}',
    'Taxa carrinho': '{:.2%}',
    'Taxa abandono': '{:.2%}'
}

# 2. Aplicação do estilo no DataFrame
tabela_visualizacao = tabela_agregado.style.format(regras_formatacao)

# 3. Exibição
tabela_visualizacao


,Nome Campanha,Gasto em Dólar,Qtd Impressões,Alcance,Qtd de clicks no Website,Qtd de procura,Qtd de visualização,Qtd adicionado ao Carrinho,Qtd de Compras,CPC,CPA,CTR,Frequência,Drop-off,Taxa carrinho,Taxa abandono,CR
0,Control Campaign,"$ 68,653.00","3,177,233","2,576,503","154,303","64,418","56,370","37,700","15,161",$ 0.44,$ 4.53,4.86%,1.23,36.53%,66.88%,59.79%,9.83%
1,Test Campaign,"$ 76,892.00","2,237,544","1,604,747","180,970","72,569","55,740","26,446","15,637",$ 0.42,$ 4.92,8.09%,1.39,30.80%,47.45%,40.87%,8.64%


Com base nos indicadores, podemos levantar algumas suspeitas acerca das duas campanhas:

1. **Métricas de custo**

    * A campanha teste obteve um custo ligeiramente menor por clique se comparado à campanha controle, indicando assim um custo menor por clique.
    * Entretanto, a campanha controle obteve um valor menor para o custo de aquisição (CPA) se comparado à campanha teste, o que indica maior eficiência neste indicador.
    * **Com base nos KPIs, vemos que a campanha teste é ligeiramente mais eficiente para atrair tráfego dado o custo da campanha, porém foi a campanha controle que foi mais eficiente para gerar vendas, sendo assim ela a otimizar os recursos financeiros empregados na campanha**

2. **Métricas de Engajamento**

    * **Com base na taxa de clique (CTR), vemos que ambas as campanhas foram efetivas**, com CTRs maiores do que 3%, entretanto destaca-se o CTR da campanha teste que chega a ser quase o dobro do valor para a campanha controle. Dessa forma, vemos que a promessa do anúncio foi atrativa para a audiência, especialmente para a campanha Teste, bem como com base no gráfico vemos que isso de deu consistentimente ao longo do mês de agosto.
    * **Para o indicador de Frequência, ambas as campanhas apresentaram uma frequência baixa**, ou seja, próxima de 1. Isso é bom, pois implica que as mesmas pessoas não estão revendo o mesmo anúncio diversas vezes, o que pode gerar um cansaço do anúncio, bem como o anúncio é exibido para mais pessoas diferentes. Entre as campanhas, foi a Controle a que obteve o menor valor de Frequência.
    * Para a taxa de *Drop-off,* vemos que a campanha Teste obteve um valor inferior à campanha controle, sendo assim retendo mais o público para visualização do site. No geral também, destaca-se que ambas as campanhas apresentaram valores bons com porcentagens menores do que 40%.
    * **Como conclusão para as Métricas de Engajamento, nota-se que a campanha Teste obteve um menor *Drop-off* e uma melhor Taxa de clique (CTR) com a campanha controle, obtendo um melhor indicador para Frequência. Entretanto, com base nos indicadores, aparentemente a Campanha Teste obteve um melhor desempenho de engajamento.**

3. **Métricas de Conversão**

    * A taxa de adição ao carrinho da campanha controle obteve um percentual maior se comparado à campanha teste, indicando, dessa forma, que a campanha foi mais efetiva para despertar o interesse do consumidor para o produto, bem como talvez os preços praticados neste momento sejam atrativos.
    * Porém, ao compararmos a taxa de carrinho da campanha controle com a taxa de abandono do produto no carrinho, denota-se que a campanha controle tem uma taxa superior à campanha teste, desse modo indicando que a mesma perde efetividade para realizar a venda do produto.
    * Destaca-se também a taxa de conversão global (CR) entre as campanhas, com a controle possuindo o valor de 9,83%, representando uma diferença de 1,19% se comparado à taxa da campanha de teste, além disso destaca-se com base no gráfico que o desempenho da campanha controle foi superior a teste, consistentimente ao longo de agosto, entretanto com picos notáveis em alguns dias. Isso indica que, mesmo a campanha controle tendo uma taxa de abandono maior, ela consegue com mais efetividade realizar a venda dos produtos para quem entrou no site. Isso, em parte, também advém do fato de que tal campanha conseguiu atender de forma mais efetiva a expectativa do consumidor ao ver o anúncio e entrar no site, exibido pela taxa de carrinho. Por conseguinte, esse talvez seja um dos principais motivos para a plataforma de anúncios privilegiar a exibição da campanha Controle, conforme demonstrado na seção 2.
    * **Como conclusões principais para as métricas de conversão, vemos que a campanha controle obteve melhor efetividade para realização da venda, entretanto a mesma poderia melhorar a taxa de abandono do carrinho, dessa forma é recomendado ao marketing analisar possíveis técnicas utilizadas na campanha de testes a fim de incorporar as mesmas à campanha controle.**

Dessa forma, analisando os três grupos de KPIs, chegamos a algumas conclusões interessantes acerca das campanhas:

* **A Campanha Controle se mostrou mais efetiva para Conversão, apresentando um custo menor para o mesmo.**
* **A Campanha Teste se mostrou mais efetiva para engajamento, desse modo, ela apresentou um menor custo por clique (CPC), entretanto não foi a campanha com maior conversão**
* **Logo, vemos que cada campanha tem uma área forte, com nenhuma delas obtendo vantagem absoluta em todas as métricas. Por isso, são recomendadas ao marketing duas opções:**
    - Empregar cada campanha de acordo com as necessidades do negócio, dessa forma empregando a campanha controle quando se busca conversões altas e a campanha teste quando se busca engajamento alto.
    - Analisar os quesitos técnicos dos quais cada campanha obteve melhores desempenhos em um conjunto de métricas e não em outro para assim conceber uma nova campanha C, onde poderia combinar tais quesitos técnicos a fim de criar uma campanha que supere em todos os grupos de métricas tanto a campanha Teste quanto a campanha Controle
***
**Entretanto, ainda deve-se verificar se estas diferenças absolutas são estatisticamente significativas, ou somente fruto do acaso. Por isso, segue-se agora para os testes de hipóteses para cada grupo de métricas. Desse modo, a fim de validar o desempenho de cada campanha.**

## 4 - Testes de Hipóteses

***
Para diferenciarmos a variação entre as taxas de cada campanha e aferir se as mesmas são fruto do acaso ou não, recorre-se a testes estatísticos. Uma vez que, assim, pode-se aferir se de fato determinada campanha obteve sucesso sobre a outra em determinado indicador.

Para isso, será utilizado o teste Z para duas proporções. A escolha do teste Z para duas proporções se dá pelos seguintes motivos:<br>
    1. Natureza agregada dos dados fornecidos<br>
    2. Amparo pelo Teorema do Limite Central, dado o grande número amostral (N)<br>
Além disso, descarta-se a utilização de teste de normalidade e heterocedasticidade, devido à agregação dos dados, uma vez que isso impossibilita o cálculo da variância individual, desse modo também exclui-se do teste o cálculo de taxas contínuas como o CPA e a Frequência

***

O teste Z se apoia em duas hipóteses, sendo elas:

Hipótese Nula (As proporções são iguais):
$H_0: p_A = p_B$

Hipótese Alternativa (As proporções são diferentes):
$H_1: p_A \neq p_B$

Se o P-Valor for **menor que 0,05, rejeitamos a Hipótese Nula**. Sendo assim, há uma diferença entre as campanhas que não pode ser explicada pelo acaso.<br>
Se o P-Valor for **maior que 0,05, não rejeitamos a Hipótese Nula**. Desse modo, não há evidências suficientes para aferir uma diferença entre as campanhas, logo isso pode ser fruto do acaso.

In [14]:
# Calculando o teste Z para as métricas

# 1. Dicionário mapeando
metricas_teste = {
    'CTR': ['Qtd de clicks no Website', 'Qtd Impressões',False],
    'Drop-off': ['Qtd de visualização', 'Qtd de clicks no Website',False],
    'Taxa carrinho': ['Qtd adicionado ao Carrinho', 'Qtd de visualização',False],
    'CR': ['Qtd de Compras', 'Qtd de clicks no Website',False],
    'Taxa abandono': ['Qtd de Compras', 'Qtd adicionado ao Carrinho', True]
}

alpha = 0.05 # Nível de significância de 5%

# 2. Loop para iterar sobre cada métrica do dicionário
for nome_metrica, colunas in metricas_teste.items():
    col_sucesso = colunas[0]
    col_tentativa = colunas[1]
    is_inversa = colunas[2]

    print(f"{'='*40}")
    print(f"Executando Teste Z para: {nome_metrica}")
    print(f"Sucesso: {col_sucesso} | Tentativa: {col_tentativa}")
    print(f"{'-'*40}")
    
    # Extração dos dados
    sucessos = np.array([
        tabela_agregado.loc[tabela_agregado['Nome Campanha'] == 'Control Campaign', col_sucesso].values[0],
        tabela_agregado.loc[tabela_agregado['Nome Campanha'] == 'Test Campaign', col_sucesso].values[0]
    ])

    tentativas = np.array([
        tabela_agregado.loc[tabela_agregado['Nome Campanha'] == 'Control Campaign', col_tentativa].values[0],
        tabela_agregado.loc[tabela_agregado['Nome Campanha'] == 'Test Campaign', col_tentativa].values[0]
    ])
    
    # Tratamento para métricas inversas (Ex: Abandono de Carrinho)
    # O "sucesso" passa a ser a quantidade de pessoas que falharam na etapa
    if is_inversa:
        sucessos = tentativas - sucessos

    # Verifica se há divisão por zero ou tentativas vazias
    if tentativas[0] == 0 or tentativas[1] == 0:
        print("Erro: Número de tentativas é zero para um dos grupos. Pulando métrica.\n")
        continue

    # Cálculo do Teste Z
    z_stat, p_valor = proportions_ztest(count=sucessos, nobs=tentativas, alternative='two-sided')
    
    # Cálculo das taxas para exibição visual
    taxa_controle = (sucessos[0] / tentativas[0]) * 100
    taxa_teste = (sucessos[1] / tentativas[1]) * 100
    
    print(f"Taxa Controle: {taxa_controle:.2f}%")
    print(f"Taxa Teste:    {taxa_teste:.2f}%")
    print(f"Estatística Z: {z_stat:.4f}")
    print(f"P-valor:       {p_valor:.4e}")
    
    # Decisão
    if p_valor < alpha:
        print("-> Conclusão: REJEITAMOS H0 (Diferença estatisticamente significativa)")
    else:
        print("-> Conclusão: NÃO REJEITAMOS H0 (Diferença pode ser obra do acaso)")
    print("\n")

Executando Teste Z para: CTR
Sucesso: Qtd de clicks no Website | Tentativa: Qtd Impressões
----------------------------------------
Taxa Controle: 4.86%
Taxa Teste:    8.09%
Estatística Z: -153.6302
P-valor:       0.0000e+00
-> Conclusão: REJEITAMOS H0 (Diferença estatisticamente significativa)


Executando Teste Z para: Drop-off
Sucesso: Qtd de visualização | Tentativa: Qtd de clicks no Website
----------------------------------------
Taxa Controle: 36.53%
Taxa Teste:    30.80%
Estatística Z: 35.0600
P-valor:       2.7436e-269
-> Conclusão: REJEITAMOS H0 (Diferença estatisticamente significativa)


Executando Teste Z para: Taxa carrinho
Sucesso: Qtd adicionado ao Carrinho | Tentativa: Qtd de visualização
----------------------------------------
Taxa Controle: 66.88%
Taxa Teste:    47.45%
Estatística Z: 65.7590
P-valor:       0.0000e+00
-> Conclusão: REJEITAMOS H0 (Diferença estatisticamente significativa)


Executando Teste Z para: CR
Sucesso: Qtd de Compras | Tentativa: Qtd de clicks

Com base nos testes anteriores, aufere-se que todas as métricas testadas apresentaram significância estatística, desse modo a diferença entre as campanhas não se da pelo acaso.<br>
Porém ressalta-se ainda que os testes, devido a presença do SRM, podem sofrer em sua confiabilidade, sendo assim exigindo cautela para afirmações causais.

## 5 - Conclusão e Recomendações
***
Com base nos resultados das seções 2, 3 e 4. Vemos que com base nas análises exploratórias cada campanha performou de forma diferente, **entretanto devido a presença de SRM a inferência causal sob a superioridade da campanha Controle ficam comprometidas**. Porém recomenda-se três frentes de ação:<br>
**Primeira Frente de Ação**: Utiliza-se as campanhas atuais dependendo do foco principal dos setores de vendas e marketing, respaldado pela análise exploratória e pelos testes de hipótese (sem confirmar causalidade), com isso utilizando cada uma das campanhas da seguinte maneira<br>

* **Controle**: Quando o foco dos setores for a Conversão<br>
    Isso se dá por tal campanha se mostrar mais efetiva para converter os anuncios em vendas, ela pode ser interessante quando queremos trazer o cliente para produtos de maior ticket médio, ou até mesmo quando queremos trazer um cliente antigo para comprar conosco. Isso poderia ser testado pela entrega de novos dados do setor de vendas e marketing a fim de aferir o valor dos itens comprados em cada campanha, caso ele se demonstre maior podemos suspeitar de tal característica.
    Contudo, ressalta-se que a presença do SRM evidencia que a plataforma de anúncio atribuiu tráfego adicional para a campanha em questão o que afeta a confiabilidade causal, para afirmar que a campanha Controle é superior em conversão devido a seu escopo criativo.

* **Teste**: Quando o foco dos setores for o Engajamento<br>
    Tal campanha foi mais efetiva para gerar engajamento, com a mesma apresentando melhores métricas relacionadas a isto, bem como destaca-se o CPC um pouco menor para a campanha o que exibe que ela teve um custo para clique menor. Recomenda-se a utilização dessa campanha quando a finalidade for atrair consumidores diferentes de uma base instalada, ou seja atrair potenciais consumidores que ainda não conhecem os nossos produtos.<br>
    Entretanto, mais uma vez, cabe ressaltar que a presença do SRM invalida a afirmação causal devido ao escopo criativo da campanha Teste.
    
**Segunda Frente de Ação**: Elaborar uma terceira campanha<br>
A elaboração de uma terceira campanha pode ser definida como uma nova possibilidade de corrigir os pontos fracos das campanhas controle e teste, uma vez que ambas não apresentaram um desempenho absoluto sobre a outra. Desse modo, recomenda-se avaliar os quesitos técnicos elaborados em ambas e fazer as alterações pelo departamento de marketing para a criação da campanha C. Após realizada e testada poderia-se averiguar o desempenho da campanha C frente as campanhas controle e teste (A e B).

**Terceira Frente de Ação**: Realizar uma nova amostragem para as campanhas A e B<br>
    Devido a presença do SRM a confiança causal entre as campanhas controle e teste ficaram com confiabilidade abalada. Com isso, recomenda-se para a equipe de marketing a criação de uma nova amostragem para as campanhas controle e teste com as opções de desabilitar a otimização automática, ou a utilização de ferramentas dedicadas a experimentação como o *Campaign Experiments do Google Ads*
